In [1]:
import torch
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
from GPT_Model import cfg, GPT
from Training_pipeline import train_model_simple, train_loader, val_loader

In [ ]:
import time
start_time = time.time()

torch.manual_seed(123)
model = GPT(cfg)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device=device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.004, weight_decay=0.1)

epochs = 10

train_losses, val_losses, tokens_seen = train_model_simple(
    model, 
    train_loader, 
    val_loader, 
    optimizer, 
    device, 
    num_epochs=epochs, 
    eval_freq=5, 
    eval_iter=5, 
    start_context="Every effort moves you", 
    tokenizer=tokenizer
    # verbose=False
)

end_time = time.time()

print("Execution time: ", (end_time-start_time)/60, "minutes")

################################################################
print("LOSS-PLOT")

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def plotlosses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))
    
    ax1.plot(epochs_seen, train_losses, label="Training Loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation Loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
    
    ax2 = ax1.twiny()
    ax2.plot(tokens_seen, train_losses, alpha=0)
    ax2.set_xlabel("Tokens seen")
    
    fig.tight_layout()
    plt.savefig("loss-plot.pdf")
    plt.show()

    
epochs_tensor = torch.linspace(0, epochs, len(train_losses))
plotlosses(epochs_tensor, tokens_seen, train_losses, val_losses)

##  Save the model

In [ ]:
torch.save(
  {"model_state_dict":model.state_dict(),
   "optimzer_state_dict":optimizer.state_dict()
   },
  "model_and_optimizer.pth"
  )

## Load the Model

In [ ]:
checkpoint=torch.load("model_and_optimizer.pth")

model=GPT(cfg)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer=torch.optim.AdamW(model.parameters(),lr=5e-4,weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimzer_state_dict"])

model.train()